# 1. Загрузка датасета

Датасет: ```https://huggingface.co/datasets/yezhengli9/wmt20-en-ru```

In [1]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import json

In [2]:
def _ok(s: str, max_len: int) -> bool:
    return bool(s) and len(s) <= max_len

def _extract_translation(ex):
    tr = ex.get("translation (translation)")
    if isinstance(tr, str):
        try:
            tr = json.loads(tr)
        except Exception:
            tr = {"en": ex.get("en", ""), "ru": ex.get("ru", "")}
    en = tr.get("en", "")
    ru = tr.get("ru", "")
    return {"src": en, "tgt": ru}


def load_wmt(max_len: int = 1000, test_size: float = 0.20, seed: int = 42):
    raw = load_dataset("yezhengli9/wmt20-en-ru")
    base = raw["train"].map(_extract_translation, remove_columns=raw["train"].column_names)
    base = base.filter(lambda ex: _ok(ex["src"], max_len) and _ok(ex["tgt"], max_len))
    
    tmp = base.shuffle(seed=seed).train_test_split(test_size=test_size, seed=seed)
    ds = DatasetDict({
        "train": tmp["train"],
        "test": tmp["test"],
    })
    return ds

ds = load_wmt()

In [3]:
print(ds)
ds['train'][0]

DatasetDict({
    train: Dataset({
        features: ['src', 'tgt'],
        num_rows: 1601
    })
    test: Dataset({
        features: ['src', 'tgt'],
        num_rows: 401
    })
})


{'src': 'At the mega "Howdy, Modi!" event last Sunday, sharing a stage with US President Donald Trump, the Prime Minister said: "They have made hatred towards India the centre of their agenda."\n',
 'tgt': 'На большом мероприятии Howdy Modi в прошлое воскресенье, выступая на одной сцене с Дональдом Трампом, премьер-министр сказал: “Они сделали ненависть к Индии центральным пунктом своей повестки”.\n'}

# 2.  Метрики

In [4]:
import evaluate

In [5]:
def compute_bleu_chrf(hyp: list[str], ref: list[str]):
    bleu = evaluate.load('bleu')
    chrf = evaluate.load('chrf')
    bleu = bleu.compute(predictions=hyp, references=ref)
    chrf = chrf.compute(predictions=hyp, references=ref)
    return {"bleu": bleu['bleu'], "chrf": chrf['score']}

# 3. Baseline

In [8]:
from transformers import MarianTokenizer, MarianMTModel
import torch

In [9]:
tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
mdl = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ru").to("cuda")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [10]:
@torch.no_grad()
def translate_batch(texts: list[str], max_new_tokens: int = 128) -> list[str]:
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True)
    enc = {k: v.to(mdl.device) for k, v in enc.items()}
    out = mdl.generate(**enc, max_new_tokens=max_new_tokens)
    return tok.batch_decode(out, skip_special_tokens=True)

N = 100 # можно изменять в зависимости от доступных ресурсов
val_src = [r["src"] for r in ds["test"]][:N]
val_ref = [r["tgt"] for r in ds["test"]][:N]

hyp = translate_batch(val_src, max_new_tokens=128)

metrics = compute_bleu_chrf(hyp, val_ref)
print({k: round(v, 2) for k, v in metrics.items()})

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'bleu': 0.2, 'chrf': 49.47}


In [11]:
print(val_src[0])
print(val_ref[0])
print(hyp[0])

Mr Agbadua argued that the allegations against Mr Sowore attracted capital punishment and that bail in such circumstances where highly restricted.

Агбадуа утверждал, что обвинения против Соворе предполагают смертную казнь, и освобождение под залог в этих условиях должно быть крайне ограничено.

Г-н Агбадуа утверждал, что обвинения в адрес г-на Совора влекут за собой смертную казнь и что залог в таких обстоятельствах является весьма ограниченным.


# 4. Обучение

In [6]:
from transformers import DataCollatorForSeq2Seq, \
                            Seq2SeqTrainingArguments, \
                            Seq2SeqTrainer, \
                            AutoTokenizer, \
                            AutoModelForSeq2SeqLM

In [7]:
def make_tokenize_fn(tok, max_src=256, max_tgt=256):
    def _fn(batch):
        src = list(batch["src"])
        tgt = list(batch["tgt"])
        
        model_inputs = tok(
            src, 
            max_length=max_src, 
            padding=False,  # Let collator handle padding
            truncation=True
        )
        
        labels = tok(
            tgt, 
            max_length=max_tgt, 
            padding=False,
            truncation=True
        )
        
        # Replace padding token id with -100 to ignore in loss
        labels_input_ids = labels["input_ids"]
        for i in range(len(labels_input_ids)):
            for j in range(len(labels_input_ids[i])):
                if labels_input_ids[i][j] == tok.pad_token_id:
                    labels_input_ids[i][j] = -100
        
        model_inputs["labels"] = labels_input_ids
        return model_inputs
    return _fn


tok = AutoTokenizer.from_pretrained("google/mt5-small")
mdl = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
tok_fn = make_tokenize_fn(tok)
train_ds  = ds["train"].map(tok_fn, batched=True, remove_columns=ds["train"].column_names)
val_ds  = ds["test"].map(tok_fn, batched=True, remove_columns=ds["test"].column_names)

collate = DataCollatorForSeq2Seq(tokenizer=tok, model=mdl)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

Map:   0%|          | 0/401 [00:00<?, ? examples/s]

In [9]:
args = Seq2SeqTrainingArguments(
    'results',
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    logging_strategy='steps',
    logging_steps=1,
    predict_with_generate=True,
    generation_max_length=128,
    report_to="none",
    bf16=True
    )

trainer = Seq2SeqTrainer(model=mdl, 
                         args=args, 
                         train_dataset=train_ds, 
                         eval_dataset=val_ds,
                         data_collator=collate)
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.435535,2.936078
2,3.893651,2.771568


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 512282880 vs 512282768

# 5. Валидация

In [14]:
res = trainer.predict(val_ds)

In [15]:
preds = res.predictions
preds[preds < 0] = 0 # trainer в качестве пэддинга ставит -100

In [16]:
hyp = [text.replace('<extra_id_0>', '').strip() 
                  for text in tok.batch_decode(preds, skip_special_tokens=True)]
val_ref = [r["tgt"] for r in ds["test"]]

In [17]:
print(compute_bleu_chrf(hyp, val_ref))

{'bleu': 0.026651698123195563, 'chrf': 20.49127852320591}


In [24]:
print(hyp[0], '\n' + val_ref[0])

В этом случае обвинение против его виновных виновных и незаконных преступлений, которые они незаконно оплачивали. 
Агбадуа утверждал, что обвинения против Соворе предполагают смертную казнь, и освобождение под залог в этих условиях должно быть крайне ограничено.

